In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import random

In [ ]:
#读数据，mat文件里变量名叫data
data=loadmat('cluster_dataset.mat')

In [ ]:
X=data['data']
x1=[i[0] for i in X]
x2=[i[1] for i in X]

In [ ]:
#先看散点图，数有几堆
plt.figure(dpi=150)
plt.scatter(x1,x2,color='b',alpha=0.8)
plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

In [ ]:
#预设聚类簇数k：请观察数据集散点图，预计聚类簇数k
k=3#我觉得3堆

In [ ]:
#随机选择数据集中k个点作为初始聚类中心
def init_centers(X,k):
    """
    输入：X:数据集(ndarray), k：预设的聚类簇数(int)
    输出：centers：初始化的中心集（建议：元素为ndarray的list)
    可能使用的函数：random.randint
    注意：需保证随机取的样本不重复
    """
    centers=[]
    used=[]
    while len(centers)<k:
        t=random.randint(0,len(X)-1)
        if t not in used:
            used.append(t)
            centers.append(np.array(X[t]))
    return centers


In [ ]:
#向量欧氏距离计算
def distance(v1,v2):
    """
    输入：v1:样本(ndarray), v2:当前对应的中心(ndarray)输出：distance:欧氏距离(float/ndarray)
    """
    d=0
    for j in range(len(v1)):
        d+=(v1[j]-v2[j])**2
    return d**0.5


In [ ]:
#将样本分配到距离最近的中心所在的簇
def cluster_assignment(X,centers):
    """
    输入：X:数据集(ndarray), centers：当前中心集（建议：元素为ndarray的list)
    输出：assignment(建议：字典：key为簇标号、value为元素是ndarray的list)
    """
    assignment={}
    for i in range(len(centers)):
        assignment[i]=[]
    for p in X:
        mind=999999
        mini=0
        for i in range(len(centers)):
            dd=distance(p,centers[i])
            if dd<mind:
                mind=dd
                mini=i
        assignment[mini].append(np.array(p))
    return assignment



In [ ]:
#代价函数
def cost_function(assignment,centers):
    """
    输入：X:数据集(ndarray), centers：当前中心，输出：cost:代价函数值（float/ndarray）
    """
    s=0
    cnt=0
    for i in range(len(centers)):
        for p in assignment[i]:
            s+=distance(p,centers[i])
            cnt+=1
    return s/cnt

In [ ]:
#更新中心
def center_update(assignment,centers):
    """
    输入：assignment, centers：当前中心，输出：new_centers:更新的中心集, stop：停机条件标识（不停机：0/停机：1）
    """
    new_centers=[]
    stop=1
    for i in range(len(centers)):
        if len(assignment[i])==0:
            new_centers.append(centers[i])
            continue
        arr=np.array(assignment[i])
        nc=np.mean(arr,axis=0)
        if nc[0]!=centers[i][0] or nc[1]!=centers[i][1]:
            stop=0
        new_centers.append(nc)
    return new_centers,stop

In [ ]:
#聚类可视化
def plot_clustering(assignment,centers,epoch):
    color=['r','b','c','g','k','y','m']
    plt.figure(dpi=150)
    for i in range(len(centers)):
        if len(assignment[i])==0:
            continue
        c=np.array(assignment[i])
        plt.scatter(c[:,0],c[:,1],c=color[i])
    for i in range(len(centers)):
        plt.scatter(centers[i][0],centers[i][1],c='k',marker='*')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('epoch'+str(epoch))
    plt.show()

In [ ]:
#kmeans及可视化
def kmeans(X,k,max_epoch,plot=True):
    """
    输入：X:数据集(ndarray), k：预设的聚类簇数(int), max_epoch：最大训练轮数(int), plot:是否可视化（True/False)
    输出：assignment, cost:最终的代价函数值，用于k-cost曲线的绘制（float/ndarray）
    """
    centers=init_centers(X,k)
    for t in range(max_epoch):
        ass=cluster_assignment(X,centers)
        if plot:
            plot_clustering(ass,centers,t)
        centers,stop=center_update(ass,centers)
        if stop==1:
            break
    ass=cluster_assignment(X,centers)
    return ass,cost_function(ass,centers)

In [ ]:
max_epoch=200

In [ ]:
assignment,_=kmeans(X,k,max_epoch)

In [ ]:
#绘制簇数-代价函数曲线：根据曲线，观察最合适的簇数k的选择
plt.figure(dpi=150)
Cost=[]
max_k=6
for ki in range(1,max_k):
    _,c=kmeans(X,ki,max_epoch,False)
    Cost.append(c)
plt.plot(range(1,max_k),Cost,c='b',marker='*')
plt.xticks(range(1,max_k))
plt.xlabel('k')
plt.ylabel('cost')
plt.title('k-cost curve')
plt.show()